In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, silhouette_score

# 1. Load and Preprocess Data
df = pd.read_csv('Country-data.csv')
features = df.drop(columns=['country'])

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# 2. Apply K-Means Clustering
km = KMeans(n_clusters=3, random_state=42)
df['KMeans_Cluster'] = km.fit_predict(features_scaled)

# 3. Apply DBSCAN for Outlier Detection
dbscan = DBSCAN(eps=1.5, min_samples=3)
df['DBSCAN_Cluster'] = dbscan.fit_predict(features_scaled)

# Map human-readable labels to clusters
cluster_map = {0: 'Developing', 1: 'Developed', 2: 'Under-developed'}
df['Development_Status'] = df['KMeans_Cluster'].map(cluster_map)
df.to_csv('segmented_country_data.csv', index=False)

# 4. Train-Test Split for Ensemble Classifiers
X = features_scaled
y = df['KMeans_Cluster']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Random Forest Ensemble Model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("--- Random Forest Classification Report ---")
print(classification_report(y_test, rf_preds, target_names=list(cluster_map.values())))

# 6. Gradient Boosting Ensemble Model
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)

print("\n--- Gradient Boosting Classification Report ---")
print(classification_report(y_test, gb_preds, target_names=list(cluster_map.values())))

# 7. Visualization
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
colors = {0: '#3498db', 1: '#2ecc71', 2: '#e74c3c'}

for cluster, group in df.groupby('KMeans_Cluster'):
    ax.scatter(group['income'], group['child_mort'],
               color=colors[cluster], label=cluster_map[cluster],
               alpha=0.7, edgecolors='w', s=100)

ax.set_title('Market Segmentation & Cluster Analysis', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Net Income Per Person (USD)', fontsize=12)
ax.set_ylabel('Child Mortality (per 1,000 live births)', fontsize=12)
ax.legend(title='Strategic Segments', fontsize=10, title_fontsize=11)
plt.tight_layout()
plt.savefig('country_segmentation_clusters.png', dpi=300)
plt.close()

--- Random Forest Classification Report ---
                 precision    recall  f1-score   support

     Developing       1.00      0.94      0.97        18
      Developed       1.00      1.00      1.00         7
Under-developed       0.90      1.00      0.95         9

       accuracy                           0.97        34
      macro avg       0.97      0.98      0.97        34
   weighted avg       0.97      0.97      0.97        34


--- Gradient Boosting Classification Report ---
                 precision    recall  f1-score   support

     Developing       1.00      0.83      0.91        18
      Developed       1.00      1.00      1.00         7
Under-developed       0.75      1.00      0.86         9

       accuracy                           0.91        34
      macro avg       0.92      0.94      0.92        34
   weighted avg       0.93      0.91      0.91        34

